# Synthetic Tracking Degradation + Robustness Framework

This notebook demonstrates the **Synthetic Tracking Degradation Framework** developed in `src/noise/degradation.py` for the Ball-Free Game State Reconstruction project.

### Objectives & Guarantees
1. **Clean Reference Baseline:** Load Metrica Sample Game 1 tracking data using canonical parser `src/data/metrica_parser.py`.
2. **Empirical Grounding:** Parameters are derived from and motivated by empirical observations in `01_metrica_tracking_quality.ipynb` and `02_metrica_motion_anomalies.ipynb`.
3. **Two-Tier Degradation Hierarchy:**
   - **Observation-Level:** Random missingness, geometric contiguous gaps, configurable coordinate jitter, and isolated coordinate jumps.
   - **Identity-Level:** Deterministic track fragmentation (`<id>_frag_<seg>`) and pairwise same-team identity switches.
4. **Controlled Severities:** Evaluates `clean`, `mild`, `moderate`, and `severe` configurations.
5. **Data Policy:** Raw tracking files in `data/raw/metrica/` remain untouched. Intermediate outputs are saved to `data/interim/degradation/` and figures to `results/figures/`.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in path
sys.path.insert(0, os.path.abspath('../../'))

from src.data.metrica_parser import load_metrica_match
from src.noise.degradation import degrade_tracking, SEVERITY_CONFIGS, ALL_DEGRADATION_NAMES, DEGRADATION_HIERARCHY

# Create designated output directories
INTERIM_DIR = '../../data/interim/degradation'
FIGURES_DIR = '../../results/figures'
os.makedirs(INTERIM_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print('Setup complete.')

## 1. Clean Data Ingestion & Reference Baseline

In [ ]:
home_path = '../../data/raw/metrica/data/Sample_Game_1/Sample_Game_1_RawTrackingData_Home_Team.csv'
away_path = '../../data/raw/metrica/data/Sample_Game_1/Sample_Game_1_RawTrackingData_Away_Team.csv'
match_id = 'sample_game_1'

players_df, ball_df = load_metrica_match(home_path, away_path, match_id)
print(f'Loaded clean tracking dataset with {len(players_df):,} player records across {players_df["frame"].nunique():,} frames.')
print(f'Unique players: {players_df["player_id"].nunique()} across teams: {list(players_df["team"].unique())}')

In [ ]:
def compute_tracking_statistics(df, label=''):
    """Compute descriptive tracking and degradation statistics."""
    total_obs = len(df)
    missing = (~df['visible']).sum()
    visible = df['visible'].sum()
    missing_pct = 100.0 * missing / total_obs if total_obs > 0 else 0.0
    visible_pct = 100.0 * visible / total_obs if total_obs > 0 else 0.0

    # Gap analysis per player track
    gap_count = 0
    gap_lengths = []
    for (team, pid), grp in df.groupby(['team', 'player_id']):
        grp_sorted = grp.sort_values('frame')
        vis = grp_sorted['visible'].values
        in_gap = False
        cur_len = 0
        for v in vis:
            if not v:
                in_gap = True
                cur_len += 1
            else:
                if in_gap:
                    gap_lengths.append(cur_len)
                    gap_count += 1
                    cur_len = 0
                    in_gap = False
        if in_gap:
            gap_lengths.append(cur_len)
            gap_count += 1

    frag_tracks = df[df['player_id'].str.contains('_frag_')]['player_id'].nunique()

    return {
        'label': label,
        'total_observations': total_obs,
        'missing_observations': int(missing),
        'missing_pct': round(missing_pct, 2),
        'visible_observations': int(visible),
        'visible_pct': round(visible_pct, 2),
        'gap_count': gap_count,
        'gap_lengths': gap_lengths,
        'gap_mean': round(float(np.mean(gap_lengths)), 1) if gap_lengths else 0.0,
        'gap_median': round(float(np.median(gap_lengths)), 1) if gap_lengths else 0.0,
        'gap_max': int(max(gap_lengths)) if gap_lengths else 0,
        'fragmented_tracks': frag_tracks,
    }

clean_stats = compute_tracking_statistics(players_df, label='clean')
print('### Clean Baseline Metrics')
for k, v in clean_stats.items():
    if k != 'gap_lengths':
        print(f'  {k}: {v}')

## 2. Empirical Grounding of Degradation Parameters

Parameters are grounded in empirical evidence from prior quality and motion anomaly audits:
- **Frame Rate:** 25 FPS ($dt = 0.04\text{ s}$) dictates temporal scaling for gap lengths and switch durations.
- **Coordinate System:** Normalised $[0.0, 1.0]$ pitch coordinates.
- **Motion Velocity Percentiles:** 99.0th percentile step displacement is $\approx 0.00288$ norm units. Coordinate jitter scales are set to $0.001$ (mild, sub-median), $0.003$ (moderate, $\approx \text{P99}$), and $0.008$ (severe, $> \text{P99.9}$).
- **Speed Anomalies & Jumps:** Isolated jump magnitudes are calibrated to $0.05 - 0.50$ norm units, exceeding normal sprinting velocity ($0.25\text{ norm/s}$) while excluding systemic half-time tracker resets (frame 71269).

## 3. Degradation Generation across Severities

We evaluate four deterministic severity configurations using explicit seeds:
- `clean` (seed 42)
- `mild` (seed 123)
- `moderate` (seed 456)
- `severe` (seed 789)

In [ ]:
SEEDS = {
    'clean': 42,
    'mild': 123,
    'moderate': 456,
    'severe': 789,
}

experiment_results = {}

for severity_name, seed_val in SEEDS.items():
    print(f'Generating {severity_name} dataset (seed={seed_val})...')
    degraded_df, metadata = degrade_tracking(
        players_df,
        severity=severity_name,
        seed=seed_val,
    )
    stats = compute_tracking_statistics(degraded_df, label=severity_name)

    # Identity changes vs clean baseline
    id_changes = (degraded_df['player_id'].values != players_df['player_id'].values).sum()
    stats['identity_changes'] = int(id_changes)

    # Synthetic jumps (spatial displacement > 0.05 norm units)
    vis_both = players_df['visible'].values & degraded_df['visible'].values
    dx = np.abs(degraded_df['x'].values - players_df['x'].values)
    dy = np.abs(degraded_df['y'].values - players_df['y'].values)
    disp = np.sqrt(np.where(vis_both, dx**2 + dy**2, 0.0))
    stats['synthetic_jumps'] = int((disp > 0.05).sum())

    # Perturbation metrics on mutually visible observations
    perturbations = disp[vis_both]
    stats['perturbation_mean'] = round(float(np.mean(perturbations)), 6) if len(perturbations) > 0 else 0.0
    stats['perturbation_std'] = round(float(np.std(perturbations)), 6) if len(perturbations) > 0 else 0.0
    stats['perturbation_max'] = round(float(np.max(perturbations)), 6) if len(perturbations) > 0 else 0.0

    experiment_results[severity_name] = {
        'df': degraded_df,
        'metadata': metadata,
        'stats': stats,
    }
    print(f'  Done: Missing {stats["missing_pct"]}%, Gaps {stats["gap_count"]}, '
          f'ID Changes {stats["identity_changes"]}, Synth Jumps {stats["synthetic_jumps"]}')

print('\nAll degradation levels generated successfully.')

## 4. Multi-Severity Degradation Summary Table

In [ ]:
summary_rows = []
for sev_name in SEEDS:
    s = experiment_results[sev_name]['stats']
    summary_rows.append({
        'Severity': s['label'].capitalize(),
        'Total Obs': f"{s['total_observations']:,}",
        'Missing %': f"{s['missing_pct']:.2f}%",
        'Visible %': f"{s['visible_pct']:.2f}%",
        'Gap Count': s['gap_count'],
        'Gap Mean': s['gap_mean'],
        'Gap Median': s['gap_median'],
        'Gap Max': s['gap_max'],
        'Perturbation Mean': s['perturbation_mean'],
        'Perturbation Max': s['perturbation_max'],
        'ID Changes': s['identity_changes'],
        'Frag Tracks': s['fragmented_tracks'],
        'Synth Jumps': s['synthetic_jumps'],
    })

summary_df = pd.DataFrame(summary_rows)
print('### Comparative Degradation Statistics')
print(summary_df.to_string(index=False))

## 5. Visualizations & Analytical Plots

In [ ]:
# 5a. Missingness Comparison
severities = list(SEEDS.keys())
missing_pcts = [experiment_results[s]['stats']['missing_pct'] for s in severities]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([s.capitalize() for s in severities], missing_pcts, color=['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728'])
ax.set_ylabel('Missing Observation Percentage (%)')
ax.set_title('Tracking Missingness across Degradation Severities')
ax.set_ylim(0, max(missing_pcts) * 1.25 if max(missing_pcts) > 0 else 1)
for bar, val in zip(bars, missing_pcts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.6,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'degradation_missingness_comparison.png'), dpi=150)
plt.show()

In [ ]:
# 5b. Gap-Length Distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
colors = ['#1f77b4', '#ff7f0e', '#d62728']

for idx, sev_name in enumerate(['mild', 'moderate', 'severe']):
    gap_lens = experiment_results[sev_name]['stats']['gap_lengths']
    ax = axes[idx]
    if gap_lens:
        ax.hist(gap_lens, bins=25, color=colors[idx], edgecolor='black', alpha=0.8)
    ax.set_title(f'{sev_name.capitalize()} Severity (n={len(gap_lens)})')
    ax.set_xlabel('Gap Duration (frames)')
    if idx == 0:
        ax.set_ylabel('Frequency Count')
    ax.grid(alpha=0.3)

plt.suptitle('Contiguous Gap Duration Distribution (Geometric Model)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'degradation_gap_distributions.png'), dpi=150)
plt.show()

In [ ]:
# 5c. Coordinate Perturbation Distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for idx, sev_name in enumerate(['mild', 'moderate', 'severe']):
    degraded = experiment_results[sev_name]['df']
    vis_both = players_df['visible'].values & degraded['visible'].values
    dx = degraded['x'].values - players_df['x'].values
    dy = degraded['y'].values - players_df['y'].values
    perturbations = np.sqrt(np.where(vis_both, dx**2 + dy**2, 0.0))
    perturbations = perturbations[vis_both & (perturbations > 0)]

    ax = axes[idx]
    if len(perturbations) > 0:
        ax.hist(perturbations, bins=40, color=colors[idx], edgecolor='black', alpha=0.8, log=True)
    ax.set_title(f'{sev_name.capitalize()} Perturbations')
    ax.set_xlabel('Displacement (norm pitch units)')
    if idx == 0:
        ax.set_ylabel('Log Count')
    ax.grid(alpha=0.3)

plt.suptitle('Coordinate Perturbation Distributions (Jitter + Jumps)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'degradation_perturbation_distributions.png'), dpi=150)
plt.show()

In [ ]:
# 5d. Clean vs Degraded Trajectory Context
example_player = players_df['player_id'].unique()[0]
example_team = players_df[players_df['player_id'] == example_player]['team'].values[0]
f_start, f_end = 1500, 1750

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for idx, sev_name in enumerate(['clean', 'mild', 'moderate', 'severe']):
    ax = axes[idx // 2][idx % 2]
    df_sev = experiment_results[sev_name]['df']

    mask = (
        (df_sev['player_id'] == example_player)
        & (df_sev['team'] == example_team)
        & (df_sev['frame'] >= f_start)
        & (df_sev['frame'] <= f_end)
        & df_sev['visible']
    )
    sub = df_sev[mask].sort_values('frame')
    if not sub.empty:
        ax.plot(sub['x'], sub['y'], 'o-', markersize=2, label=f'Player {example_player}', alpha=0.85)

    ax.set_title(f'{sev_name.capitalize()} Trajectory (Frames {f_start}-{f_end})')
    ax.set_xlabel('Pitch X')
    ax.set_ylabel('Pitch Y')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Trajectory Perturbation Context for Player {example_player}', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'degradation_trajectory_comparison.png'), dpi=150)
plt.show()

In [ ]:
# 5e. Track Fragmentation Example
severe_df = experiment_results['severe']['df']
frag_ids = severe_df[severe_df['player_id'].str.contains('_frag_')]['player_id'].unique()

if len(frag_ids) > 0:
    base_pid = frag_ids[0].split('_frag_')[0]
    related_ids = [fid for fid in frag_ids if fid.startswith(f'{base_pid}_frag_')]
    frag_team = severe_df[severe_df['player_id'] == related_ids[0]]['team'].values[0]

    fig, ax = plt.subplots(figsize=(10, 5))

    # Clean trajectory for comparison
    clean_mask = (players_df['player_id'] == base_pid) & (players_df['team'] == frag_team) & players_df['visible']
    clean_sub = players_df[clean_mask].sort_values('frame')
    ax.plot(clean_sub['x'], clean_sub['y'], 'k--', linewidth=1, alpha=0.3, label=f'Clean Reference ({base_pid})')

    seg_colors = plt.cm.Set1(np.linspace(0, 1, len(related_ids)))
    for fid, col in zip(related_ids, seg_colors):
        f_sub = severe_df[(severe_df['player_id'] == fid) & severe_df['visible']].sort_values('frame')
        ax.plot(f_sub['x'], f_sub['y'], 'o-', markersize=2, color=col, alpha=0.75, label=f'Fragment {fid}')

    ax.set_title(f'Synthetic Track Fragmentation: Player {base_pid} (Severe Configuration)')
    ax.set_xlabel('Pitch X')
    ax.set_ylabel('Pitch Y')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'degradation_fragmentation_example.png'), dpi=150)
    plt.show()
else:
    print('No track fragmentation found in this execution seed.')

## 6. Serialization of Interim Artifacts & Metadata

In [ ]:
for sev_name, res in experiment_results.items():
    # Save JSON experiment metadata
    meta_filepath = os.path.join(INTERIM_DIR, f'metadata_{sev_name}.json')
    with open(meta_filepath, 'w') as f:
        json.dump(res['metadata'], f, indent=2)

    # Save parquet degraded dataset in ignored interim folder
    parquet_filepath = os.path.join(INTERIM_DIR, f'degraded_{sev_name}.parquet')
    res['df'].to_parquet(parquet_filepath, index=False)
    print(f'Persisted interim artifacts for [{sev_name}]: {meta_filepath}, {parquet_filepath}')

print('\nSerialization complete. No raw data was modified.')